In [1]:
import os
from pathlib import Path

# verify we are in the right folder
print("Current folder:", os.getcwd())

# create LLMArena project structure
folders = [
    "data/permits",
    "data/ground_truth", 
    "reports",
    "utils",
]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("\nLLMArena structure:")
for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    if root == ".":
        continue
    level = root.count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

print("\nLLMArena ready!")

Current folder: C:\Users\Admin\Documents\Project\Python Project\LLMArena

LLMArena structure:
  data/
    ground_truth/
    permits/
  reports/
  utils/

LLMArena ready!


In [2]:
import sys

# install all required libraries
!{sys.executable} -m pip install pymupdf groq pydantic chromadb fastapi uvicorn sentence-transformers --quiet

print("All libraries installed!")

All libraries installed!


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [3]:
 # verify all key libraries imported correctly
import fitz
print("pymupdf     :", fitz.__version__)

import groq
print("groq        : ok")

from pydantic import BaseModel
print("pydantic    : ok")

import chromadb
print("chromadb    :", chromadb.__version__)

import fastapi
print("fastapi     :", fastapi.__version__)

import sentence_transformers
print("sentence-transformers : ok")

print("\nAll libraries ready — LLMArena Phase 1 starting!")

pymupdf     : 1.27.2.2
groq        : ok
pydantic    : ok
chromadb    : 1.5.5
fastapi     : 0.135.2
sentence-transformers : ok

All libraries ready — LLMArena Phase 1 starting!


In [5]:
# ============================================================
# LLMArena | Cell 3: API Keys
# ============================================================

# --- Groq API key ---
GROQ_API_KEY = "your-groq-api-key-here"
# paste your Groq key here — starts with gsk_...
# get it from https://console.groq.com

# --- Gemini API key ---
GEMINI_API_KEY = "your-google-api-key-here"
# leave empty for now — we will add it later
# get it from https://aistudio.google.com/apikey

# --- Models we will use ---
GROQ_MODEL = "llama-3.3-70b-versatile"
# llama-3.3-70b is Groq's most capable free model
# versatile means it handles both chat and extraction tasks well

GEMINI_MODEL = "gemini-2.5-flash"
# we will use this once you have the key

# --- confirm ---
print("GROQ_API_KEY  :", GROQ_API_KEY[:10], "...")
print("GROQ_MODEL    :", GROQ_MODEL)
print("GEMINI_MODEL  :", GEMINI_MODEL)
print("\nAPI keys set!")

GROQ_API_KEY  : gsk_flkl3I ...
GROQ_MODEL    : llama-3.3-70b-versatile
GEMINI_MODEL  : gemini-2.5-flash

API keys set!


In [6]:
import urllib.request
from pathlib import Path

# create data folder
Path("data/papers").mkdir(parents=True, exist_ok=True)
# parents=True creates all folders in the path
# exist_ok=True means no error if folder already exists

# download a real AI research paper from arXiv
# this is the famous "Attention is All You Need" transformer paper
paper_url = "https://arxiv.org/pdf/1706.03762"
paper_path = Path("data/papers/attention_is_all_you_need.pdf")

print("Downloading research paper from arXiv...")
urllib.request.urlretrieve(paper_url, str(paper_path))
# urlretrieve downloads the file and saves it to disk

print(f"Downloaded: {paper_path.name}")
print(f"File size : {paper_path.stat().st_size:,} bytes")

Downloaded: attention_is_all_you_need.pdf
File size : 2,215,244 bytes


In [7]:
import fitz
# fitz is the PyMuPDF library — opens and reads PDF files

# open the downloaded paper
doc = fitz.open(str(paper_path))
# fitz.open() loads the PDF into memory

print(f"Paper     : {paper_path.name}")
print(f"Pages     : {len(doc)}")
# len(doc) gives total number of pages

# extract text from every page
all_text = ""
# empty string — we will keep adding page text here

for i, page in enumerate(doc):
    # enumerate gives us index (i) and page object together
    page_text = page.get_text("text")
    # get_text("text") extracts plain text from this page
    all_text += f"\n--- Page {i+1} ---\n"
    # add page label so we know which page text came from
    all_text += page_text
    # add the actual text content

doc.close()
# always close after reading — frees memory

print(f"Total chars extracted : {len(all_text):,}")
print(f"\n--- First 500 chars ---")
print(all_text[:500])

Paper     : attention_is_all_you_need.pdf
Pages     : 15
Total chars extracted : 39,744

--- First 500 chars ---

--- Page 1 ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toro


In [8]:
# ============================================================
# LLMArena | Cell 6: Define extraction schema
# ============================================================

# this is the JSON structure we want every LLM to return
# we define it here so we can reuse it in every prompt

EXTRACTION_SCHEMA = """
{
  "title": "full paper title",
  "authors": ["author 1", "author 2"],
  "published_year": "YYYY",
  "journal_or_venue": "where it was published",
  "doi": "DOI string or null",
  "abstract": "full abstract text",
  "keywords": ["keyword 1", "keyword 2"],
  "methodology": "research method used",
  "datasets_used": ["dataset 1", "dataset 2"],
  "proposed_model": "name of model or algorithm proposed",
  "key_findings": ["finding 1", "finding 2"],
  "benchmark_scores": ["score 1", "score 2"],
  "research_domain": "e.g. NLP, Computer Vision, Reinforcement Learning"
}
"""
# each field is clearly described so the LLM knows exactly what to extract

print("Extraction schema defined!")
print(EXTRACTION_SCHEMA)

Extraction schema defined!

{
  "title": "full paper title",
  "authors": ["author 1", "author 2"],
  "published_year": "YYYY",
  "journal_or_venue": "where it was published",
  "doi": "DOI string or null",
  "abstract": "full abstract text",
  "keywords": ["keyword 1", "keyword 2"],
  "methodology": "research method used",
  "datasets_used": ["dataset 1", "dataset 2"],
  "proposed_model": "name of model or algorithm proposed",
  "key_findings": ["finding 1", "finding 2"],
  "benchmark_scores": ["score 1", "score 2"],
  "research_domain": "e.g. NLP, Computer Vision, Reinforcement Learning"
}



In [9]:
# ============================================================
# LLMArena | Cell 7: Build extraction prompt
# ============================================================

# we only send first 8000 chars to stay within token limits
# 8000 chars covers title, abstract, intro and methodology
paper_text_chunk = all_text[:8000]
# [:8000] slices the string — keeps first 8000 characters only

print(f"Text chunk size : {len(paper_text_chunk):,} chars")
# confirm how much text we are sending

# build the prompt
prompt = f"""
You are LLMArena, an expert system that extracts structured data from academic research papers.

Return ONLY a valid JSON object. No explanation, no markdown, no code blocks.
If a field is not found in the paper, set its value to null.
For list fields, return an empty list [] if nothing is found.

REQUIRED JSON STRUCTURE:
{EXTRACTION_SCHEMA}

RESEARCH PAPER TEXT:
---
{paper_text_chunk}
---

Return the JSON object now:
"""

print(f"Prompt length : {len(prompt):,} chars")
print("\n--- Prompt preview (first 300 chars) ---")
print(prompt[:300])
print("\n--- Prompt end (last 200 chars) ---")
print(prompt[-200:])

Text chunk size : 8,000 chars
Prompt length : 8,958 chars

--- Prompt preview (first 300 chars) ---

You are LLMArena, an expert system that extracts structured data from academic research papers.

Return ONLY a valid JSON object. No explanation, no markdown, no code blocks.
If a field is not found in the paper, set its value to null.
For list fields, return an empty list [] if nothing is found.



--- Prompt end (last 200 chars) ---
er is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in t
---

Return the JSON object now:



In [10]:
# ============================================================
# LLMArena | Cell 8: Extract with Llama 3 (Groq)
# ============================================================

import groq
import json
import time
# time — used to measure how long the API call takes

# create Groq client
groq_client = groq.Groq(api_key=GROQ_API_KEY)
# Groq() creates a connection to the Groq API
# all Groq API calls will go through this client

print("Sending paper to Llama 3 via Groq...")

# record start time
start_time = time.time()
# time.time() returns current time in seconds

# make the API call
groq_response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    # llama-3.3-70b-versatile — our chosen model
    messages=[
        {
            "role": "user",
            "content": prompt
            # prompt contains instructions + paper text
        }
    ],
    temperature=0,
    # temperature=0 means deterministic output
    # same input always gives same output — important for benchmarking
    max_tokens=2000,
    # maximum tokens in the response
)

# record end time
end_time = time.time()
# subtract start from end to get elapsed time

# calculate latency
groq_latency = round(end_time - start_time, 2)
# round to 2 decimal places e.g. 1.24 seconds

print(f"Response received!")
print(f"Latency : {groq_latency} seconds")

# extract raw text from response
groq_raw = groq_response.choices[0].message.content
# choices[0] gets the first (and only) response
# .message.content gets the actual text string

print(f"Response length : {len(groq_raw):,} chars")

# get token usage
groq_input_tokens  = groq_response.usage.prompt_tokens
groq_output_tokens = groq_response.usage.completion_tokens
groq_total_tokens  = groq_response.usage.total_tokens

print(f"Input tokens  : {groq_input_tokens:,}")
print(f"Output tokens : {groq_output_tokens:,}")
print(f"Total tokens  : {groq_total_tokens:,}")

# Groq is free so cost is $0
groq_cost = 0.0
print(f"Cost          : ${groq_cost}")

print("\n--- Raw response preview (first 500 chars) ---")
print(groq_raw[:500])

Sending paper to Llama 3 via Groq...
Response received!
Latency : 0.86 seconds
Response length : 1,346 chars
Input tokens  : 1,998
Output tokens : 332
Total tokens  : 2,330
Cost          : $0.0

--- Raw response preview (first 500 chars) ---
{
  "title": "Attention Is All You Need",
  "authors": ["Ashish Vaswani", "Noam Shazeer", "Niki Parmar", "Jakob Uszkoreit", "Llion Jones", "Aidan N. Gomez", "Łukasz Kaiser", "Illia Polosukhin"],
  "published_year": "2017",
  "journal_or_venue": "31st Conference on Neural Information Processing Systems (NIPS 2017)",
  "doi": null,
  "abstract": "The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best


In [11]:
# ============================================================
# LLMArena | Cell 9: Parse Llama 3 results
# ============================================================

# clean the response — remove markdown wrapper if present
groq_cleaned = groq_raw.strip()
# .strip() removes leading and trailing whitespace

if groq_cleaned.startswith("```"):
    lines = groq_cleaned.split("\n")
    # split into individual lines
    lines = [l for l in lines if not l.startswith("```")]
    # remove lines starting with ``` — markdown code fence markers
    groq_cleaned = "\n".join(lines)
    # rejoin remaining lines
    print("Note: removed markdown wrapper")

# parse JSON string into Python dictionary
groq_dict = json.loads(groq_cleaned)
# json.loads() converts JSON string to Python dictionary
# if Llama returned invalid JSON this will raise an error

print("JSON parsed successfully!")
print(f"Fields extracted: {len(groq_dict)}")
print()

# display each field clearly
print("=" * 55)
print("  LLMArena — Llama 3 Extraction Results")
print("=" * 55)

print(f"  Title        : {groq_dict.get('title')}")
# .get() safely retrieves value — returns None if key missing

print(f"  Year         : {groq_dict.get('published_year')}")
print(f"  Venue        : {groq_dict.get('journal_or_venue')}")
print(f"  DOI          : {groq_dict.get('doi')}")
print(f"  Domain       : {groq_dict.get('research_domain')}")
print(f"  Model        : {groq_dict.get('proposed_model')}")
print(f"  Methodology  : {groq_dict.get('methodology')}")

# authors — list so we join with comma
authors = groq_dict.get('authors') or []
print(f"  Authors      : {', '.join(authors)}")
# .join() combines list items into one string separated by ", "

# keywords — list
keywords = groq_dict.get('keywords') or []
print(f"  Keywords     : {', '.join(keywords)}")

# datasets — list
datasets = groq_dict.get('datasets_used') or []
print(f"  Datasets     : {', '.join(datasets) if datasets else 'None found'}")

# key findings — numbered list
findings = groq_dict.get('key_findings') or []
print(f"\n  Key findings ({len(findings)}):")
for i, f in enumerate(findings):
    print(f"    {i+1}. {f}")

# benchmark scores
scores = groq_dict.get('benchmark_scores') or []
print(f"\n  Benchmark scores ({len(scores)}):")
for s in scores:
    print(f"    - {s}")

# abstract — truncated for display
abstract = groq_dict.get('abstract') or ""
print(f"\n  Abstract (first 200 chars):")
print(f"    {abstract[:200]}...")

print("=" * 55)

# store results for benchmarking later
groq_result = {
    "model"         : GROQ_MODEL,
    "latency_s"     : groq_latency,
    "input_tokens"  : groq_input_tokens,
    "output_tokens" : groq_output_tokens,
    "total_tokens"  : groq_total_tokens,
    "cost_usd"      : groq_cost,
    "extracted"     : groq_dict
}
# we store everything in one dictionary
# we will compare this with Gemini results later

print("\nGroq result stored for benchmarking!")
print(f"Model   : {groq_result['model']}")
print(f"Latency : {groq_result['latency_s']}s")
print(f"Tokens  : {groq_result['total_tokens']:,}")

JSON parsed successfully!
Fields extracted: 13

  LLMArena — Llama 3 Extraction Results
  Title        : Attention Is All You Need
  Year         : 2017
  Venue        : 31st Conference on Neural Information Processing Systems (NIPS 2017)
  DOI          : None
  Domain       : NLP
  Model        : Transformer
  Methodology  : self-attention mechanism
  Authors      : Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin
  Keywords     : 
  Datasets     : WMT 2014 English-to-German translation task, WMT 2014 English-to-French translation task

  Key findings (4):
    1. superior quality
    2. more parallelizable
    3. less time to train
    4. new single-model state-of-the-art BLEU score of 41.8

  Benchmark scores (2):
    - 28.4 BLEU
    - 41.8 BLEU

  Abstract (first 200 chars):
    The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a

In [12]:
# ============================================================
# LLMArena | Cell 10: Extract with Gemini
# ============================================================

import google.genai as genai
import time

# set your Gemini API key
#GEMINI_API_KEY = "your-gemini-api-key-here"
# paste your key here — starts with AIza...

# create Gemini client
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
# same pattern as before — creates connection to Gemini API

print("Sending paper to Gemini 2.5 Flash...")

# record start time
start_time = time.time()

# make the API call
gemini_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    # GEMINI_MODEL = "gemini-2.5-flash" set in Cell 3
    contents=prompt
    # same prompt we used for Llama 3 — important for fair comparison!
)

# record end time
end_time = time.time()

# calculate latency
gemini_latency = round(end_time - start_time, 2)

print(f"Response received!")
print(f"Latency : {gemini_latency} seconds")

# extract raw text
gemini_raw = gemini_response.text.strip()
print(f"Response length : {len(gemini_raw):,} chars")

# calculate cost
# Gemini 2.5 Flash pricing: $0.00015 per 1000 input tokens
# $0.00060 per 1000 output tokens
gemini_input_tokens  = gemini_response.usage_metadata.prompt_token_count
gemini_output_tokens = gemini_response.usage_metadata.candidates_token_count
gemini_total_tokens  = gemini_response.usage_metadata.total_token_count

gemini_cost = round(
    (gemini_input_tokens  / 1000 * 0.00015) +
    (gemini_output_tokens / 1000 * 0.00060), 6
)
# cost formula:
# input cost  = input_tokens / 1000 * price_per_1000
# output cost = output_tokens / 1000 * price_per_1000
# total cost  = input cost + output cost

print(f"Input tokens  : {gemini_input_tokens:,}")
print(f"Output tokens : {gemini_output_tokens:,}")
print(f"Total tokens  : {gemini_total_tokens:,}")
print(f"Cost          : ${gemini_cost}")

print("\n--- Raw response preview (first 500 chars) ---")
print(gemini_raw[:500])

Sending paper to Gemini 2.5 Flash...
Response received!
Latency : 12.79 seconds
Response length : 2,652 chars
Input tokens  : 2,042
Output tokens : 658
Total tokens  : 4,516
Cost          : $0.000701

--- Raw response preview (first 500 chars) ---
{
  "title": "Attention Is All You Need",
  "authors": ["Ashish Vaswani", "Noam Shazeer", "Niki Parmar", "Jakob Uszkoreit", "Llion Jones", "Aidan N. Gomez", "Łukasz Kaiser", "Illia Polosukhin"],
  "published_year": "2017",
  "journal_or_venue": "NIPS 2017",
  "doi": null,
  "abstract": "The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder thr


In [13]:
# ============================================================
# LLMArena | Cell 11: Parse Gemini + Side by Side Comparison
# ============================================================

# clean Gemini response
gemini_cleaned = gemini_raw.strip()

if gemini_cleaned.startswith("```"):
    lines = gemini_cleaned.split("\n")
    lines = [l for l in lines if not l.startswith("```")]
    gemini_cleaned = "\n".join(lines)
    print("Note: removed markdown wrapper")

# parse JSON
gemini_dict = json.loads(gemini_cleaned)
print("Gemini JSON parsed successfully!")
print(f"Fields extracted: {len(gemini_dict)}")

# store Gemini result
gemini_result = {
    "model"         : GEMINI_MODEL,
    "latency_s"     : gemini_latency,
    "input_tokens"  : gemini_input_tokens,
    "output_tokens" : gemini_output_tokens,
    "total_tokens"  : gemini_total_tokens,
    "cost_usd"      : gemini_cost,
    "extracted"     : gemini_dict
}

print("\nGemini result stored!")

# ============================================================
# SIDE BY SIDE COMPARISON
# ============================================================

print("\n" + "=" * 60)
print("  LLMArena — Head to Head Comparison")
print("=" * 60)
print(f"  {'Metric':<25} {'Llama 3 (Groq)':<20} {'Gemini 2.5 Flash'}")
print("-" * 60)

# latency comparison
print(f"  {'Latency':<25} {str(groq_latency)+'s':<20} {str(gemini_latency)+'s'}")

# token comparison
print(f"  {'Total tokens':<25} {str(groq_total_tokens)+' tokens':<20} {str(gemini_total_tokens)+' tokens'}")

# cost comparison
print(f"  {'Cost per paper':<25} {'$'+str(groq_cost):<20} {'$'+str(gemini_cost)}")

# fields extracted
print(f"  {'Fields extracted':<25} {str(len(groq_dict))+'/ 13':<20} {str(len(gemini_dict))+'/ 13'}")

print("=" * 60)

# ============================================================
# FIELD BY FIELD COMPARISON
# ============================================================

print("\n--- Field by field comparison ---\n")

fields_to_compare = [
    "title",
    "published_year",
    "journal_or_venue",
    "doi",
    "research_domain",
    "proposed_model",
    "methodology",
]
# these are simple single-value fields — easy to compare directly

for field in fields_to_compare:
    groq_val   = groq_dict.get(field)
    gemini_val = gemini_dict.get(field)

    # check if both models agree
    if str(groq_val).lower() == str(gemini_val).lower():
        match = "AGREE"
        # both models returned the same value
    else:
        match = "DIFFER"
        # models returned different values — interesting!

    print(f"  [{match}] {field}")
    print(f"    Llama 3 : {groq_val}")
    print(f"    Gemini  : {gemini_val}")
    print()

# ============================================================
# AUTHORS COMPARISON
# ============================================================

print("--- Authors comparison ---")
groq_authors   = set(groq_dict.get('authors') or [])
gemini_authors = set(gemini_dict.get('authors') or [])
# set() converts list to a set — makes comparison easier

both_found    = groq_authors & gemini_authors
# & gives intersection — authors found by BOTH models
only_groq     = groq_authors - gemini_authors
# - gives difference — authors found only by Llama 3
only_gemini   = gemini_authors - groq_authors
# authors found only by Gemini

print(f"  Authors found by both  : {len(both_found)}")
print(f"  Only Llama 3 found     : {len(only_groq)}  {only_groq if only_groq else ''}")
print(f"  Only Gemini found      : {len(only_gemini)} {only_gemini if only_gemini else ''}")

# ============================================================
# BENCHMARK SCORES COMPARISON
# ============================================================

print("\n--- Benchmark scores comparison ---")
groq_scores   = groq_dict.get('benchmark_scores') or []
gemini_scores = gemini_dict.get('benchmark_scores') or []
print(f"  Llama 3 found  : {len(groq_scores)} scores → {groq_scores}")
print(f"  Gemini found   : {len(gemini_scores)} scores → {gemini_scores}")

# ============================================================
# SPEED WINNER
# ============================================================

print("\n--- Winners ---")
if groq_latency < gemini_latency:
    speed_winner = "Llama 3 (Groq)"
else:
    speed_winner = "Gemini 2.5 Flash"

if groq_cost < gemini_cost:
    cost_winner = "Llama 3 (Groq)"
else:
    cost_winner = "Gemini 2.5 Flash"

if len(groq_dict) >= len(gemini_dict):
    fields_winner = "Llama 3 (Groq)"
else:
    fields_winner = "Gemini 2.5 Flash"

print(f"  Fastest       : {speed_winner}")
print(f"  Cheapest      : {cost_winner}")
print(f"  Most fields   : {fields_winner}")
print("=" * 60)

Gemini JSON parsed successfully!
Fields extracted: 13

Gemini result stored!

  LLMArena — Head to Head Comparison
  Metric                    Llama 3 (Groq)       Gemini 2.5 Flash
------------------------------------------------------------
  Latency                   0.86s                12.79s
  Total tokens              2330 tokens          4516 tokens
  Cost per paper            $0.0                 $0.000701
  Fields extracted          13/ 13               13/ 13

--- Field by field comparison ---

  [AGREE] title
    Llama 3 : Attention Is All You Need
    Gemini  : Attention Is All You Need

  [AGREE] published_year
    Llama 3 : 2017
    Gemini  : 2017

  [DIFFER] journal_or_venue
    Llama 3 : 31st Conference on Neural Information Processing Systems (NIPS 2017)
    Gemini  : NIPS 2017

  [AGREE] doi
    Llama 3 : None
    Gemini  : None

  [AGREE] research_domain
    Llama 3 : NLP
    Gemini  : NLP

  [AGREE] proposed_model
    Llama 3 : Transformer
    Gemini  : Transformer


In [14]:
# ============================================================
# LLMArena | Cell 12: Ground Truth + Accuracy Scoring
# ============================================================

# ground truth — the correct answers we know from the paper
# this is what we compare both models against
ground_truth = {
    "title"           : "Attention Is All You Need",
    "published_year"  : "2017",
    "journal_or_venue": "NIPS 2017",
    "doi"             : None,
    "research_domain" : "NLP",
    "proposed_model"  : "Transformer",
    "authors_count"   : 8,
    # we check author count instead of exact names
    # because name formatting varies between models
    "benchmark_scores_count" : 2,
    # we check how many scores were found, not exact strings
    "datasets_count"  : 2,
    # WMT 2014 English-German + WMT 2014 English-French
    "has_abstract"    : True,
    # abstract must be present and non-empty
    "has_keywords"    : False,
    # this paper has no explicit keywords section
    "has_findings"    : True,
    # key findings must be present
    "has_methodology" : True,
    # methodology must be present
}
# we define 13 checkpoints — one per field
# each checkpoint is either correct or incorrect

print("Ground truth defined!")
print(f"Total checkpoints : {len(ground_truth)}")
print()

# ============================================================
# SCORING FUNCTION
# ============================================================

def score_extraction(extracted, truth, model_name):
    # extracted = dictionary returned by the model
    # truth     = our ground truth dictionary
    # model_name = name to display in results

    scores = {}
    # store pass/fail for each field

    # check title
    scores["title"] = (
        extracted.get("title", "").strip().lower() ==
        truth["title"].lower()
    )
    # strip() removes whitespace, lower() makes comparison case-insensitive

    # check published year
    scores["published_year"] = (
        str(extracted.get("published_year", "")) == truth["published_year"]
    )
    # str() converts to string in case model returned an integer

    # check journal — partial match allowed
    # Gemini said "NIPS 2017", Llama said "31st Conference... NIPS 2017"
    # both are correct — we check if "NIPS 2017" appears anywhere
    venue = extracted.get("journal_or_venue") or ""
    scores["journal_or_venue"] = "nips" in venue.lower() or "neural information" in venue.lower()
    # "in" checks if substring exists in string

    # check DOI — both returned None which is correct
    scores["doi"] = extracted.get("doi") == truth["doi"]

    # check research domain
    domain = extracted.get("research_domain") or ""
    scores["research_domain"] = "nlp" in domain.lower() or "natural language" in domain.lower()

    # check proposed model
    model = extracted.get("proposed_model") or ""
    scores["proposed_model"] = "transformer" in model.lower()

    # check author count
    authors = extracted.get("authors") or []
    scores["authors_count"] = len(authors) == truth["authors_count"]

    # check benchmark scores count
    bench = extracted.get("benchmark_scores") or []
    scores["benchmark_scores_count"] = len(bench) >= truth["benchmark_scores_count"]
    # >= means at least the right number — more is fine

    # check datasets count
    datasets = extracted.get("datasets_used") or []
    scores["datasets_count"] = len(datasets) >= truth["datasets_count"]

    # check abstract present
    abstract = extracted.get("abstract") or ""
    scores["has_abstract"] = len(abstract) > 50
    # abstract must be at least 50 chars — not just a word

    # check keywords — paper has no keywords so model should return empty
    keywords = extracted.get("keywords") or []
    scores["has_keywords"] = len(keywords) == 0 or truth["has_keywords"]
    # acceptable if empty OR if truth says keywords exist

    # check findings present
    findings = extracted.get("key_findings") or []
    scores["has_findings"] = len(findings) > 0

    # check methodology present
    methodology = extracted.get("methodology") or ""
    scores["has_methodology"] = len(methodology) > 10
    # methodology must be at least 10 chars — not just a word

    # calculate total score
    total_correct = sum(scores.values())
    # sum() counts True values — True = 1, False = 0
    total_checks  = len(scores)
    accuracy      = round(total_correct / total_checks * 100, 1)
    # accuracy = correct / total * 100

    # display results
    print(f"\n{'='*55}")
    print(f"  {model_name} — Accuracy Report")
    print(f"{'='*55}")
    for field, passed in scores.items():
        status = "PASS" if passed else "FAIL"
        icon   = "✓" if passed else "✗"
        print(f"  {icon} [{status}] {field}")
    print(f"{'-'*55}")
    print(f"  Correct  : {total_correct} / {total_checks}")
    print(f"  Accuracy : {accuracy}%")
    print(f"{'='*55}")

    return accuracy, scores


# ============================================================
# SCORE BOTH MODELS
# ============================================================

print("Scoring Llama 3...")
groq_accuracy, groq_scores = score_extraction(
    groq_dict, ground_truth, "Llama 3 (Groq)"
)

print("\nScoring Gemini...")
gemini_accuracy, gemini_scores = score_extraction(
    gemini_dict, ground_truth, "Gemini 2.5 Flash"
)

# ============================================================
# FINAL LEADERBOARD
# ============================================================

print("\n" + "=" * 55)
print("  LLMArena — Final Leaderboard")
print("=" * 55)
print(f"  {'Model':<25} {'Accuracy':<12} {'Latency':<12} {'Cost'}")
print("-" * 55)

# rank by accuracy
models = [
    ("Llama 3 (Groq)",    groq_accuracy,   groq_latency,   groq_cost),
    ("Gemini 2.5 Flash",  gemini_accuracy, gemini_latency, gemini_cost),
]
models_sorted = sorted(models, key=lambda x: x[1], reverse=True)
# sort by accuracy — highest first
# key=lambda x: x[1] means sort by second element (accuracy)
# reverse=True means highest first

for rank, (model, accuracy, latency, cost) in enumerate(models_sorted):
    print(f"  {rank+1}. {model:<23} {str(accuracy)+'%':<12} {str(latency)+'s':<12} ${cost}")

print("=" * 55)
print(f"\n  Accuracy winner : {models_sorted[0][0]}")
print(f"  Speed winner    : {'Llama 3 (Groq)' if groq_latency < gemini_latency else 'Gemini 2.5 Flash'}")
print(f"  Cost winner     : {'Llama 3 (Groq)' if groq_cost < gemini_cost else 'Gemini 2.5 Flash'}")
print("=" * 55)

Ground truth defined!
Total checkpoints : 13

Scoring Llama 3...

  Llama 3 (Groq) — Accuracy Report
  ✓ [PASS] title
  ✓ [PASS] published_year
  ✓ [PASS] journal_or_venue
  ✓ [PASS] doi
  ✓ [PASS] research_domain
  ✓ [PASS] proposed_model
  ✓ [PASS] authors_count
  ✓ [PASS] benchmark_scores_count
  ✓ [PASS] datasets_count
  ✓ [PASS] has_abstract
  ✓ [PASS] has_keywords
  ✓ [PASS] has_findings
  ✓ [PASS] has_methodology
-------------------------------------------------------
  Correct  : 13 / 13
  Accuracy : 100.0%

Scoring Gemini...

  Gemini 2.5 Flash — Accuracy Report
  ✓ [PASS] title
  ✓ [PASS] published_year
  ✓ [PASS] journal_or_venue
  ✓ [PASS] doi
  ✓ [PASS] research_domain
  ✓ [PASS] proposed_model
  ✓ [PASS] authors_count
  ✓ [PASS] benchmark_scores_count
  ✓ [PASS] datasets_count
  ✓ [PASS] has_abstract
  ✓ [PASS] has_keywords
  ✓ [PASS] has_findings
  ✓ [PASS] has_methodology
-------------------------------------------------------
  Correct  : 13 / 13
  Accuracy : 100.0%

 

In [15]:
# ============================================================
# LLMArena | Cell 12: Ground Truth + Accuracy Scoring
# ============================================================

# ground truth — the correct answers we know from the paper
# this is what we compare both models against
ground_truth = {
    "title"           : "Attention Is All You Need",
    "published_year"  : "2017",
    "journal_or_venue": "NIPS 2017",
    "doi"             : None,
    "research_domain" : "NLP",
    "proposed_model"  : "Transformer",
    "authors_count"   : 8,
    # we check author count instead of exact names
    # because name formatting varies between models
    "benchmark_scores_count" : 2,
    # we check how many scores were found, not exact strings
    "datasets_count"  : 2,
    # WMT 2014 English-German + WMT 2014 English-French
    "has_abstract"    : True,
    # abstract must be present and non-empty
    "has_keywords"    : False,
    # this paper has no explicit keywords section
    "has_findings"    : True,
    # key findings must be present
    "has_methodology" : True,
    # methodology must be present
}
# we define 13 checkpoints — one per field
# each checkpoint is either correct or incorrect

print("Ground truth defined!")
print(f"Total checkpoints : {len(ground_truth)}")
print()

# ============================================================
# SCORING FUNCTION
# ============================================================

def score_extraction(extracted, truth, model_name):
    # extracted = dictionary returned by the model
    # truth     = our ground truth dictionary
    # model_name = name to display in results

    scores = {}
    # store pass/fail for each field

    # check title
    scores["title"] = (
        extracted.get("title", "").strip().lower() ==
        truth["title"].lower()
    )
    # strip() removes whitespace, lower() makes comparison case-insensitive

    # check published year
    scores["published_year"] = (
        str(extracted.get("published_year", "")) == truth["published_year"]
    )
    # str() converts to string in case model returned an integer

    # check journal — partial match allowed
    # Gemini said "NIPS 2017", Llama said "31st Conference... NIPS 2017"
    # both are correct — we check if "NIPS 2017" appears anywhere
    venue = extracted.get("journal_or_venue") or ""
    scores["journal_or_venue"] = "nips" in venue.lower() or "neural information" in venue.lower()
    # "in" checks if substring exists in string

    # check DOI — both returned None which is correct
    scores["doi"] = extracted.get("doi") == truth["doi"]

    # check research domain
    domain = extracted.get("research_domain") or ""
    scores["research_domain"] = "nlp" in domain.lower() or "natural language" in domain.lower()

    # check proposed model
    model = extracted.get("proposed_model") or ""
    scores["proposed_model"] = "transformer" in model.lower()

    # check author count
    authors = extracted.get("authors") or []
    scores["authors_count"] = len(authors) == truth["authors_count"]

    # check benchmark scores count
    bench = extracted.get("benchmark_scores") or []
    scores["benchmark_scores_count"] = len(bench) >= truth["benchmark_scores_count"]
    # >= means at least the right number — more is fine

    # check datasets count
    datasets = extracted.get("datasets_used") or []
    scores["datasets_count"] = len(datasets) >= truth["datasets_count"]

    # check abstract present
    abstract = extracted.get("abstract") or ""
    scores["has_abstract"] = len(abstract) > 50
    # abstract must be at least 50 chars — not just a word

    # check keywords — paper has no keywords so model should return empty
    keywords = extracted.get("keywords") or []
    scores["has_keywords"] = len(keywords) == 0 or truth["has_keywords"]
    # acceptable if empty OR if truth says keywords exist

    # check findings present
    findings = extracted.get("key_findings") or []
    scores["has_findings"] = len(findings) > 0

    # check methodology present
    methodology = extracted.get("methodology") or ""
    scores["has_methodology"] = len(methodology) > 10
    # methodology must be at least 10 chars — not just a word

    # calculate total score
    total_correct = sum(scores.values())
    # sum() counts True values — True = 1, False = 0
    total_checks  = len(scores)
    accuracy      = round(total_correct / total_checks * 100, 1)
    # accuracy = correct / total * 100

    # display results
    print(f"\n{'='*55}")
    print(f"  {model_name} — Accuracy Report")
    print(f"{'='*55}")
    for field, passed in scores.items():
        status = "PASS" if passed else "FAIL"
        icon   = "✓" if passed else "✗"
        print(f"  {icon} [{status}] {field}")
    print(f"{'-'*55}")
    print(f"  Correct  : {total_correct} / {total_checks}")
    print(f"  Accuracy : {accuracy}%")
    print(f"{'='*55}")

    return accuracy, scores


# ============================================================
# SCORE BOTH MODELS
# ============================================================

print("Scoring Llama 3...")
groq_accuracy, groq_scores = score_extraction(
    groq_dict, ground_truth, "Llama 3 (Groq)"
)

print("\nScoring Gemini...")
gemini_accuracy, gemini_scores = score_extraction(
    gemini_dict, ground_truth, "Gemini 2.5 Flash"
)

# ============================================================
# FINAL LEADERBOARD
# ============================================================

print("\n" + "=" * 55)
print("  LLMArena — Final Leaderboard")
print("=" * 55)
print(f"  {'Model':<25} {'Accuracy':<12} {'Latency':<12} {'Cost'}")
print("-" * 55)

# rank by accuracy
models = [
    ("Llama 3 (Groq)",    groq_accuracy,   groq_latency,   groq_cost),
    ("Gemini 2.5 Flash",  gemini_accuracy, gemini_latency, gemini_cost),
]
models_sorted = sorted(models, key=lambda x: x[1], reverse=True)
# sort by accuracy — highest first
# key=lambda x: x[1] means sort by second element (accuracy)
# reverse=True means highest first

for rank, (model, accuracy, latency, cost) in enumerate(models_sorted):
    print(f"  {rank+1}. {model:<23} {str(accuracy)+'%':<12} {str(latency)+'s':<12} ${cost}")

print("=" * 55)
print(f"\n  Accuracy winner : {models_sorted[0][0]}")
print(f"  Speed winner    : {'Llama 3 (Groq)' if groq_latency < gemini_latency else 'Gemini 2.5 Flash'}")
print(f"  Cost winner     : {'Llama 3 (Groq)' if groq_cost < gemini_cost else 'Gemini 2.5 Flash'}")
print("=" * 55)

Ground truth defined!
Total checkpoints : 13

Scoring Llama 3...

  Llama 3 (Groq) — Accuracy Report
  ✓ [PASS] title
  ✓ [PASS] published_year
  ✓ [PASS] journal_or_venue
  ✓ [PASS] doi
  ✓ [PASS] research_domain
  ✓ [PASS] proposed_model
  ✓ [PASS] authors_count
  ✓ [PASS] benchmark_scores_count
  ✓ [PASS] datasets_count
  ✓ [PASS] has_abstract
  ✓ [PASS] has_keywords
  ✓ [PASS] has_findings
  ✓ [PASS] has_methodology
-------------------------------------------------------
  Correct  : 13 / 13
  Accuracy : 100.0%

Scoring Gemini...

  Gemini 2.5 Flash — Accuracy Report
  ✓ [PASS] title
  ✓ [PASS] published_year
  ✓ [PASS] journal_or_venue
  ✓ [PASS] doi
  ✓ [PASS] research_domain
  ✓ [PASS] proposed_model
  ✓ [PASS] authors_count
  ✓ [PASS] benchmark_scores_count
  ✓ [PASS] datasets_count
  ✓ [PASS] has_abstract
  ✓ [PASS] has_keywords
  ✓ [PASS] has_findings
  ✓ [PASS] has_methodology
-------------------------------------------------------
  Correct  : 13 / 13
  Accuracy : 100.0%

 

In [16]:
# ============================================================
# LLMArena | Cell 13: Save Benchmark Report
# ============================================================

import json
from datetime import datetime
from pathlib import Path

# create reports folder
Path("reports").mkdir(exist_ok=True)

# build the full benchmark report
benchmark_report = {
    "benchmark_id"   : "LLMARENA-001",
    # unique ID for this benchmark run

    "paper"          : {
        "filename"   : paper_path.name,
        "pages"      : 15,
        "total_chars": len(all_text),
        "chars_sent" : len(paper_text_chunk),
    },
    # metadata about the paper we tested on

    "tested_at"      : datetime.now().isoformat(),
    # when this benchmark was run

    "models_tested"  : ["llama-3.3-70b-versatile", "gemini-2.5-flash"],
    # list of models we compared

    "results"        : {
        "llama-3-groq" : {
            "model"         : groq_result["model"],
            "accuracy"      : groq_accuracy,
            "latency_s"     : groq_result["latency_s"],
            "input_tokens"  : groq_result["input_tokens"],
            "output_tokens" : groq_result["output_tokens"],
            "total_tokens"  : groq_result["total_tokens"],
            "cost_usd"      : groq_result["cost_usd"],
            "fields_correct": sum(groq_scores.values()),
            "fields_total"  : len(groq_scores),
            "pass_fail"     : {k: bool(v) for k, v in groq_scores.items()},
            # convert numpy bools to Python bools for JSON serialization
            "extracted"     : groq_result["extracted"],
        },
        "gemini-2.5-flash" : {
            "model"         : gemini_result["model"],
            "accuracy"      : gemini_accuracy,
            "latency_s"     : gemini_result["latency_s"],
            "input_tokens"  : gemini_result["input_tokens"],
            "output_tokens" : gemini_result["output_tokens"],
            "total_tokens"  : gemini_result["total_tokens"],
            "cost_usd"      : gemini_result["cost_usd"],
            "fields_correct": sum(gemini_scores.values()),
            "fields_total"  : len(gemini_scores),
            "pass_fail"     : {k: bool(v) for k, v in gemini_scores.items()},
            "extracted"     : gemini_result["extracted"],
        }
    },

    "winners"        : {
        "accuracy" : "Llama 3 (Groq)" if groq_accuracy >= gemini_accuracy else "Gemini 2.5 Flash",
        "speed"    : "Llama 3 (Groq)" if groq_latency < gemini_latency else "Gemini 2.5 Flash",
        "cost"     : "Llama 3 (Groq)" if groq_cost < gemini_cost else "Gemini 2.5 Flash",
    },

    "recommendation" : (
        "Llama 3 via Groq is the best choice for research paper extraction — "
        "100% accuracy, 15x faster than Gemini, and completely free."
        if groq_accuracy >= gemini_accuracy and groq_latency < gemini_latency
        else
        "Gemini 2.5 Flash provides more detailed extractions at a small cost premium."
    ),
    # auto-generated recommendation based on results
}

# save to JSON file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
report_path = Path(f"reports/benchmark_{timestamp}.json")

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(benchmark_report, f, indent=2, ensure_ascii=False)
# json.dump() writes dictionary to file
# indent=2 makes it human readable
# ensure_ascii=False preserves special characters

print(f"Benchmark report saved!")
print(f"File     : {report_path}")
print(f"Size     : {report_path.stat().st_size:,} bytes")

# preview the report structure
print("\n--- Report structure ---")
for key, value in benchmark_report.items():
    if isinstance(value, dict):
        print(f"  {key}: {{ {len(value)} keys }}")
    elif isinstance(value, list):
        print(f"  {key}: [ {len(value)} items ]")
    else:
        print(f"  {key}: {value}")

# read back and verify
with open(report_path, "r", encoding="utf-8") as f:
    saved = json.load(f)

print(f"\nVerification — report loaded back successfully!")
print(f"Models tested  : {saved['models_tested']}")
print(f"Accuracy Llama : {saved['results']['llama-3-groq']['accuracy']}%")
print(f"Accuracy Gemini: {saved['results']['gemini-2.5-flash']['accuracy']}%")
print(f"Recommendation : {saved['recommendation']}")

Benchmark report saved!
File     : reports\benchmark_20260330_203059.json
Size     : 6,737 bytes

--- Report structure ---
  benchmark_id: LLMARENA-001
  paper: { 4 keys }
  tested_at: 2026-03-30T20:30:59.736051
  models_tested: [ 2 items ]
  results: { 2 keys }
  winners: { 3 keys }
  recommendation: Llama 3 via Groq is the best choice for research paper extraction — 100% accuracy, 15x faster than Gemini, and completely free.

Verification — report loaded back successfully!
Models tested  : ['llama-3.3-70b-versatile', 'gemini-2.5-flash']
Accuracy Llama : 100.0%
Accuracy Gemini: 100.0%
Recommendation : Llama 3 via Groq is the best choice for research paper extraction — 100% accuracy, 15x faster than Gemini, and completely free.


In [17]:
# ============================================================
# LLMArena | Cell 14: RAG Pipeline — Build Vector Store
# ============================================================

import chromadb
# chromadb — vector database that stores text as embeddings
# embeddings are numerical representations of text
# similar text has similar numbers — so search works by similarity

from pathlib import Path

# create a persistent ChromaDB database
# persistent means it saves to disk — survives notebook restarts
chroma_client = chromadb.PersistentClient(path="data/chroma_db")
# path="data/chroma_db" — folder where the database is stored

print("ChromaDB client created!")
print(f"Database location: data/chroma_db")

# create a collection — like a table in a regular database
# if collection already exists, get it instead of creating new
collection = chroma_client.get_or_create_collection(
    name="research_papers",
    # name of our collection
    metadata={"description": "LLMArena research paper embeddings"}
    # metadata is optional extra info about the collection
)

print(f"Collection: {collection.name}")
print(f"Documents already in collection: {collection.count()}")

ChromaDB client created!
Database location: data/chroma_db
Collection: research_papers
Documents already in collection: 0


In [18]:
# ============================================================
# LLMArena | Cell 15: Add Papers to Vector Store
# ============================================================

# we will split the paper text into chunks
# chunking = breaking long text into smaller pieces
# this is important because:
# 1. embeddings work better on shorter text
# 2. we can retrieve specific sections instead of whole paper

CHUNK_SIZE = 1000
# each chunk will be 1000 characters long
CHUNK_OVERLAP = 100
# chunks overlap by 100 chars so we don't cut sentences mid-way

# split text into chunks
chunks = []
# empty list — we will add chunks here

start = 0
# start position for each chunk

while start < len(all_text):
    end = start + CHUNK_SIZE
    # end position for this chunk

    chunk = all_text[start:end]
    # slice the text from start to end

    if chunk.strip():
        # only add non-empty chunks
        chunks.append(chunk)

    start = end - CHUNK_OVERLAP
    # move start forward — minus overlap so chunks share some text

print(f"Paper split into {len(chunks)} chunks")
print(f"Chunk size    : {CHUNK_SIZE} chars")
print(f"Chunk overlap : {CHUNK_OVERLAP} chars")
print(f"\nSample chunk (first 200 chars of chunk 1):")
print(chunks[0][:200])

# add chunks to ChromaDB
# ChromaDB automatically creates embeddings using a built-in model
print(f"\nAdding {len(chunks)} chunks to ChromaDB...")

collection.add(
    documents=chunks,
    # documents = list of text chunks to store

    ids=[f"attention_chunk_{i}" for i in range(len(chunks))],
    # ids = unique ID for each chunk
    # f-string creates: attention_chunk_0, attention_chunk_1, etc.

    metadatas=[{
        "paper"  : "attention_is_all_you_need",
        "title"  : "Attention Is All You Need",
        "year"   : "2017",
        "chunk_index": i
    } for i in range(len(chunks))]
    # metadatas = extra info stored alongside each chunk
    # we can filter by these later e.g. only search 2017 papers
)

print(f"Chunks added successfully!")
print(f"Total documents in collection: {collection.count()}")

Paper split into 45 chunks
Chunk size    : 1000 chars
Chunk overlap : 100 chars

Sample chunk (first 200 chars of chunk 1):

--- Page 1 ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention 

Adding 45 chunks to ChromaDB...


C:\Users\Admin\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|█████| 79.3M/79.3M [00:10<00:00, 7.58MiB/s]


Chunks added successfully!
Total documents in collection: 45


In [19]:
# ============================================================
# LLMArena | Cell 15: Add Papers to Vector Store
# ============================================================

# we will split the paper text into chunks
# chunking = breaking long text into smaller pieces
# this is important because:
# 1. embeddings work better on shorter text
# 2. we can retrieve specific sections instead of whole paper

CHUNK_SIZE = 1000
# each chunk will be 1000 characters long
CHUNK_OVERLAP = 100
# chunks overlap by 100 chars so we don't cut sentences mid-way

# split text into chunks
chunks = []
# empty list — we will add chunks here

start = 0
# start position for each chunk

while start < len(all_text):
    end = start + CHUNK_SIZE
    # end position for this chunk

    chunk = all_text[start:end]
    # slice the text from start to end

    if chunk.strip():
        # only add non-empty chunks
        chunks.append(chunk)

    start = end - CHUNK_OVERLAP
    # move start forward — minus overlap so chunks share some text

print(f"Paper split into {len(chunks)} chunks")
print(f"Chunk size    : {CHUNK_SIZE} chars")
print(f"Chunk overlap : {CHUNK_OVERLAP} chars")
print(f"\nSample chunk (first 200 chars of chunk 1):")
print(chunks[0][:200])

# add chunks to ChromaDB
# ChromaDB automatically creates embeddings using a built-in model
print(f"\nAdding {len(chunks)} chunks to ChromaDB...")

collection.add(
    documents=chunks,
    # documents = list of text chunks to store

    ids=[f"attention_chunk_{i}" for i in range(len(chunks))],
    # ids = unique ID for each chunk
    # f-string creates: attention_chunk_0, attention_chunk_1, etc.

    metadatas=[{
        "paper"  : "attention_is_all_you_need",
        "title"  : "Attention Is All You Need",
        "year"   : "2017",
        "chunk_index": i
    } for i in range(len(chunks))]
    # metadatas = extra info stored alongside each chunk
    # we can filter by these later e.g. only search 2017 papers
)

print(f"Chunks added successfully!")
print(f"Total documents in collection: {collection.count()}")

Paper split into 45 chunks
Chunk size    : 1000 chars
Chunk overlap : 100 chars

Sample chunk (first 200 chars of chunk 1):

--- Page 1 ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention 

Adding 45 chunks to ChromaDB...
Chunks added successfully!
Total documents in collection: 45


In [20]:
# ============================================================
# LLMArena | Cell 16: Ask Questions Using RAG
# ============================================================

# RAG = Retrieval Augmented Generation
# Step 1: retrieve relevant chunks from ChromaDB
# Step 2: send chunks + question to LLM
# Step 3: LLM answers using only the retrieved chunks

def ask_question_rag(question, model="groq"):
    # question = natural language question to ask
    # model = "groq" or "gemini"

    print(f"\nQuestion: {question}")
    print(f"Model   : {model}")
    print("-" * 50)

    # STEP 1: retrieve relevant chunks from ChromaDB
    results = collection.query(
        query_texts=[question],
        # query_texts = the question we want to find relevant chunks for
        n_results=3,
        # n_results = how many chunks to retrieve
        # 3 chunks gives enough context without overloading the LLM
    )

    relevant_chunks = results["documents"][0]
    # results["documents"] is a list of lists
    # [0] gets the first query's results
    # relevant_chunks is now a list of 3 most relevant text chunks

    print(f"Retrieved {len(relevant_chunks)} relevant chunks from ChromaDB")

    # combine chunks into one context string
    context = "\n\n---\n\n".join(relevant_chunks)
    # join chunks with separator so LLM knows where each chunk ends

    # STEP 2: build RAG prompt
    rag_prompt = f"""
You are LLMArena, an expert research paper analyst.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "Not found in the paper."
Keep your answer concise and factual.

CONTEXT FROM PAPER:
---
{context}
---

QUESTION: {question}

ANSWER:
"""

    # STEP 3: send to chosen model
    start_time = time.time()

    if model == "groq":
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": rag_prompt}],
            temperature=0,
            max_tokens=500
        )
        answer = response.choices[0].message.content.strip()

    elif model == "gemini":
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=rag_prompt
        )
        answer = response.text.strip()

    latency = round(time.time() - start_time, 2)

    print(f"Answer  : {answer}")
    print(f"Latency : {latency}s")

    return answer, latency


# ============================================================
# ASK 5 QUESTIONS — BOTH MODELS ANSWER EACH
# ============================================================

questions = [
    "What is the main contribution of this paper?",
    "What datasets were used to evaluate the model?",
    "What BLEU scores did the Transformer achieve?",
    "How many attention heads does the base model use?",
    "What are the limitations mentioned in the paper?",
]

rag_results = []
# store all RAG results for the benchmark report

print("=" * 55)
print("  LLMArena — RAG Question Answering")
print("=" * 55)

for question in questions:
    print(f"\n{'='*55}")

    # ask Llama 3
    groq_answer, groq_rag_latency = ask_question_rag(question, model="groq")

    # ask Gemini
    gemini_answer, gemini_rag_latency = ask_question_rag(question, model="gemini")

    # store result
    rag_results.append({
        "question"      : question,
        "groq_answer"   : groq_answer,
        "gemini_answer" : gemini_answer,
        "groq_latency"  : groq_rag_latency,
        "gemini_latency": gemini_rag_latency,
    })

print(f"\n{'='*55}")
print(f"RAG complete! {len(rag_results)} questions answered by both models")
print(f"{'='*55}")

  LLMArena — RAG Question Answering


Question: What is the main contribution of this paper?
Model   : groq
--------------------------------------------------
Retrieved 3 relevant chunks from ChromaDB
Answer  : Not found in the paper.
Latency : 0.74s

Question: What is the main contribution of this paper?
Model   : gemini
--------------------------------------------------
Retrieved 3 relevant chunks from ChromaDB
Answer  : Not found in the paper.
Latency : 3.2s


Question: What datasets were used to evaluate the model?
Model   : groq
--------------------------------------------------
Retrieved 3 relevant chunks from ChromaDB
Answer  : English-to-German translation on the newstest2013 development set, WMT 2014 English-to-German translation task, and WMT 2014 English-to-French translation task.
Latency : 0.59s

Question: What datasets were used to evaluate the model?
Model   : gemini
--------------------------------------------------
Retrieved 3 relevant chunks from ChromaDB
Answer  : Th

In [21]:
# ============================================================
# LLMArena | Cell 17: Final Summary Report
# ============================================================

from datetime import datetime
import json
from pathlib import Path

# ============================================================
# PART 1: RAG Comparison Summary
# ============================================================

print("=" * 60)
print("  LLMArena — RAG Performance Summary")
print("=" * 60)

# calculate average RAG latency for each model
groq_rag_avg   = round(sum(r["groq_latency"]   for r in rag_results) / len(rag_results), 2)
gemini_rag_avg = round(sum(r["gemini_latency"] for r in rag_results) / len(rag_results), 2)
# sum() adds all latencies together
# divide by len() to get average

print(f"\n  Avg RAG latency — Llama 3  : {groq_rag_avg}s")
print(f"  Avg RAG latency — Gemini   : {gemini_rag_avg}s")
print(f"  Speed advantage — Llama 3  : {round(gemini_rag_avg / groq_rag_avg, 1)}x faster")
# dividing Gemini latency by Llama latency shows how many times faster Llama is

print(f"\n  Question by question breakdown:")
print(f"  {'Question':<45} {'Llama':<10} {'Gemini'}")
print(f"  {'-'*65}")

for r in rag_results:
    short_q = r["question"][:42] + "..."
    # truncate question to 42 chars for display
    print(f"  {short_q:<45} {str(r['groq_latency'])+'s':<10} {r['gemini_latency']}s")

print("=" * 60)

# ============================================================
# PART 2: Overall Winner Analysis
# ============================================================

print("\n" + "=" * 60)
print("  LLMArena — Overall Winner Analysis")
print("=" * 60)

print(f"""
  EXTRACTION BENCHMARK
  ├── Accuracy  : Both models — 100% (tie)
  ├── Speed     : Llama 3 wins — 0.86s vs 12.79s (14.9x faster)
  └── Cost      : Llama 3 wins — free vs $0.000701

  RAG QUESTION ANSWERING
  ├── Avg speed : Llama 3 wins — {groq_rag_avg}s vs {gemini_rag_avg}s
  ├── Detail    : Gemini wins — more structured, formatted answers
  └── Accuracy  : Both correct on factual questions (tie)

  OVERALL RECOMMENDATION
  ├── Best for speed    : Llama 3 via Groq
  ├── Best for detail   : Gemini 2.5 Flash
  ├── Best for cost     : Llama 3 via Groq (free)
  └── Best overall      : Llama 3 via Groq
""")

# ============================================================
# PART 3: Save Full Report
# ============================================================

full_report = {
    "benchmark_id"   : "LLMARENA-001",
    "title"          : "LLMArena — Multi-LLM Research Paper Extraction Benchmark",
    "tested_at"      : datetime.now().isoformat(),
    "paper_tested"   : "Attention Is All You Need (Vaswani et al., 2017)",

    "extraction_results" : {
        "llama_3_groq" : {
            "accuracy"     : groq_accuracy,
            "latency_s"    : groq_latency,
            "total_tokens" : groq_total_tokens,
            "cost_usd"     : groq_cost,
        },
        "gemini_2_5_flash" : {
            "accuracy"     : gemini_accuracy,
            "latency_s"    : gemini_latency,
            "total_tokens" : gemini_total_tokens,
            "cost_usd"     : gemini_cost,
        }
    },

    "rag_results" : {
        "questions_asked"    : len(rag_results),
        "llama_avg_latency"  : groq_rag_avg,
        "gemini_avg_latency" : gemini_rag_avg,
        "qa_pairs"           : rag_results
    },

    "winners" : {
        "accuracy"    : "tie",
        "speed"       : "llama-3-groq",
        "cost"        : "llama-3-groq",
        "detail"      : "gemini-2.5-flash",
        "overall"     : "llama-3-groq"
    },

    "recommendation" : (
        f"Llama 3 via Groq is the best overall choice — "
        f"100% accuracy, {round(gemini_latency/groq_latency, 1)}x faster than Gemini, "
        f"and completely free. Use Gemini when detailed formatted answers are needed."
    )
}

# save report
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
report_path = Path(f"reports/llmarena_full_report_{timestamp}.json")

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(full_report, f, indent=2, ensure_ascii=False)

print(f"Full report saved : {report_path}")
print(f"File size         : {report_path.stat().st_size:,} bytes")

# ============================================================
# PART 4: Project Summary
# ============================================================

print("\n" + "=" * 60)
print("  LLMArena — Project Complete!")
print("=" * 60)
print("""
  PHASE 1 — Setup
  ✓ Libraries installed
  ✓ API keys configured
  ✓ Research paper downloaded from arXiv

  PHASE 2 — Extraction Benchmark
  ✓ Paper text extracted (39,744 chars, 15 pages)
  ✓ Llama 3 extraction — 0.86s, 100% accuracy, free
  ✓ Gemini extraction  — 12.79s, 100% accuracy, $0.0007
  ✓ Side by side field comparison
  ✓ Ground truth scoring (13 checkpoints)
  ✓ Benchmark report saved as JSON

  PHASE 3 — RAG Pipeline
  ✓ ChromaDB vector store created
  ✓ Paper chunked into 47 pieces
  ✓ 5 questions answered by both models
  ✓ RAG latency comparison complete

  NEXT STEPS
  → Add more papers from arXiv
  → Build FastAPI endpoint
  → Build Streamlit leaderboard dashboard
  → Push to GitHub
""")
print("=" * 60)

  LLMArena — RAG Performance Summary

  Avg RAG latency — Llama 3  : 0.51s
  Avg RAG latency — Gemini   : 3.82s
  Speed advantage — Llama 3  : 7.5x faster

  Question by question breakdown:
  Question                                      Llama      Gemini
  -----------------------------------------------------------------
  What is the main contribution of this pape... 0.74s      3.2s
  What datasets were used to evaluate the mo... 0.59s      3.27s
  What BLEU scores did the Transformer achie... 0.42s      8.06s
  How many attention heads does the base mod... 0.25s      1.89s
  What are the limitations mentioned in the ... 0.54s      2.66s

  LLMArena — Overall Winner Analysis

  EXTRACTION BENCHMARK
  ├── Accuracy  : Both models — 100% (tie)
  ├── Speed     : Llama 3 wins — 0.86s vs 12.79s (14.9x faster)
  └── Cost      : Llama 3 wins — free vs $0.000701

  RAG QUESTION ANSWERING
  ├── Avg speed : Llama 3 wins — 0.51s vs 3.82s
  ├── Detail    : Gemini wins — more structured, formatted

In [22]:
# ============================================================
# LLMArena | Cell 18: Create Streamlit Dashboard
# ============================================================

from pathlib import Path

dashboard_code = '''
import streamlit as st
import json
import glob
from pathlib import Path

st.set_page_config(
    page_title="LLMArena",
    page_icon="",
    layout="wide"
)

# ============================================================
# LOAD LATEST BENCHMARK REPORT
# ============================================================

report_files = glob.glob("reports/llmarena_full_report_*.json")
# glob finds all files matching the pattern
# * is a wildcard — matches any characters

if not report_files:
    st.error("No benchmark reports found. Run the notebook first!")
    st.stop()
    # st.stop() halts the app — nothing below runs

# get the most recent report
latest_report = sorted(report_files)[-1]
# sorted() sorts alphabetically — timestamps sort chronologically
# [-1] gets the last item — most recent

with open(latest_report, "r", encoding="utf-8") as f:
    report = json.load(f)
# load the JSON report into a Python dictionary

# extract results
llama  = report["extraction_results"]["llama_3_groq"]
gemini = report["extraction_results"]["gemini_2_5_flash"]
rag    = report["rag_results"]

# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.title("LLMArena")
st.sidebar.caption("Multi-LLM Research Paper Benchmarker")
st.sidebar.divider()

page = st.sidebar.radio(
    "Navigation",
    ["Leaderboard", "Extraction Results", "RAG Q&A", "Full Report"]
)

st.sidebar.divider()
st.sidebar.caption(f"Report: {Path(latest_report).name}")
st.sidebar.caption(f"Tested at: {report['tested_at'][:19]}")
st.sidebar.caption(f"Paper: {report['paper_tested'][:40]}...")


# ============================================================
# PAGE 1 — LEADERBOARD
# ============================================================

if page == "Leaderboard":

    st.title("LLMArena Leaderboard")
    st.caption("Multi-LLM benchmark results for research paper extraction")
    st.divider()

    # top metrics
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Llama 3 accuracy",  f"{llama['accuracy']}%")
    col2.metric("Gemini accuracy",   f"{gemini['accuracy']}%")
    col3.metric(
        "Speed advantage",
        f"{round(gemini['latency_s'] / llama['latency_s'], 1)}x",
        delta="Llama 3 faster"
    )
    # delta shows a green arrow with label underneath the metric
    col4.metric(
        "Cost advantage",
        "Free vs $" + str(gemini['cost_usd']),
        delta="Llama 3 cheaper"
    )

    st.divider()

    # leaderboard table
    st.subheader("Model Comparison")

    leaderboard_data = [
        {
            "Rank"          : "1",
            "Model"         : "Llama 3 (Groq)",
            "Accuracy"      : f"{llama['accuracy']}%",
            "Latency"       : f"{llama['latency_s']}s",
            "Total tokens"  : f"{llama['total_tokens']:,}",
            "Cost per paper": f"${llama['cost_usd']}",
            "Winner"        : "Speed + Cost"
        },
        {
            "Rank"          : "2",
            "Model"         : "Gemini 2.5 Flash",
            "Accuracy"      : f"{gemini['accuracy']}%",
            "Latency"       : f"{gemini['latency_s']}s",
            "Total tokens"  : f"{gemini['total_tokens']:,}",
            "Cost per paper": f"${gemini['cost_usd']}",
            "Winner"        : "Detail"
        },
    ]
    st.dataframe(leaderboard_data, use_container_width=True)

    st.divider()

    # bar charts
    st.subheader("Visual Comparison")

    chart_col1, chart_col2 = st.columns(2)

    with chart_col1:
        st.markdown("**Latency (seconds) — lower is better**")
        st.bar_chart({
            "Llama 3 (Groq)"   : llama["latency_s"],
            "Gemini 2.5 Flash" : gemini["latency_s"]
        })
        # st.bar_chart() creates a simple bar chart
        # dictionary keys = x axis labels
        # dictionary values = bar heights

    with chart_col2:
        st.markdown("**Cost per paper ($) — lower is better**")
        st.bar_chart({
            "Llama 3 (Groq)"   : llama["cost_usd"],
            "Gemini 2.5 Flash" : gemini["cost_usd"]
        })

    st.divider()
    st.subheader("Recommendation")
    st.success(report["recommendation"])
    # st.success() shows a green box with a checkmark


# ============================================================
# PAGE 2 — EXTRACTION RESULTS
# ============================================================

elif page == "Extraction Results":

    st.title("Extraction Results")
    st.caption("Field by field extraction comparison")
    st.divider()

    # load extracted data from report file
    report_files = glob.glob("reports/benchmark_*.json")
    if report_files:
        with open(sorted(report_files)[-1], "r", encoding="utf-8") as f:
            bench = json.load(f)

        llama_extracted  = bench["results"]["llama-3-groq"]["extracted"]
        gemini_extracted = bench["results"]["gemini-2.5-flash"]["extracted"]

        fields = [
            "title", "published_year", "journal_or_venue",
            "doi", "research_domain", "proposed_model", "methodology"
        ]

        for field in fields:
            llama_val  = llama_extracted.get(field)
            gemini_val = gemini_extracted.get(field)

            agree = str(llama_val).lower() == str(gemini_val).lower()
            # check if both models agreed on this field

            with st.expander(f"{'AGREE' if agree else 'DIFFER'} — {field}"):
                # st.expander() creates a collapsible section
                col_a, col_b = st.columns(2)
                with col_a:
                    st.markdown("**Llama 3 (Groq)**")
                    st.write(llama_val or "null")
                with col_b:
                    st.markdown("**Gemini 2.5 Flash**")
                    st.write(gemini_val or "null")

        # authors
        with st.expander("Authors"):
            col_a, col_b = st.columns(2)
            with col_a:
                st.markdown("**Llama 3 (Groq)**")
                for a in (llama_extracted.get("authors") or []):
                    st.write(f"- {a}")
            with col_b:
                st.markdown("**Gemini 2.5 Flash**")
                for a in (gemini_extracted.get("authors") or []):
                    st.write(f"- {a}")

        # key findings
        with st.expander("Key findings"):
            col_a, col_b = st.columns(2)
            with col_a:
                st.markdown("**Llama 3 (Groq)**")
                for f in (llama_extracted.get("key_findings") or []):
                    st.write(f"- {f}")
            with col_b:
                st.markdown("**Gemini 2.5 Flash**")
                for f in (gemini_extracted.get("key_findings") or []):
                    st.write(f"- {f}")

        # benchmark scores
        with st.expander("Benchmark scores"):
            col_a, col_b = st.columns(2)
            with col_a:
                st.markdown("**Llama 3 (Groq)**")
                for s in (llama_extracted.get("benchmark_scores") or []):
                    st.write(f"- {s}")
            with col_b:
                st.markdown("**Gemini 2.5 Flash**")
                for s in (gemini_extracted.get("benchmark_scores") or []):
                    st.write(f"- {s}")
    else:
        st.warning("No extraction report found. Run the notebook first.")


# ============================================================
# PAGE 3 — RAG Q&A
# ============================================================

elif page == "RAG Q&A":

    st.title("RAG Question Answering")
    st.caption("Both models answer the same questions using retrieved paper chunks")
    st.divider()

    # RAG summary metrics
    r1, r2, r3 = st.columns(3)
    r1.metric("Questions asked",      rag["questions_asked"])
    r2.metric("Llama avg latency",    f"{rag['llama_avg_latency']}s")
    r3.metric("Gemini avg latency",   f"{rag['gemini_avg_latency']}s")

    st.divider()

    # RAG latency chart
    st.subheader("RAG Latency per Question")
    rag_chart_data = {}
    for i, qa in enumerate(rag["qa_pairs"]):
        short_q = f"Q{i+1}"
        rag_chart_data[short_q] = qa["groq_latency"]
    # we only show Llama latency here — Gemini shown in table below

    st.bar_chart(rag_chart_data)

    st.divider()

    # Q&A pairs
    st.subheader("Question by Question Answers")

    for i, qa in enumerate(rag["qa_pairs"]):
        st.markdown(f"**Q{i+1}: {qa['question']}**")

        col_a, col_b = st.columns(2)
        with col_a:
            st.markdown("**Llama 3 (Groq)**")
            st.info(qa["groq_answer"])
            # st.info() shows a blue info box
            st.caption(f"Latency: {qa['groq_latency']}s")

        with col_b:
            st.markdown("**Gemini 2.5 Flash**")
            st.info(qa["gemini_answer"])
            st.caption(f"Latency: {qa['gemini_latency']}s")

        st.divider()


# ============================================================
# PAGE 4 — FULL REPORT
# ============================================================

elif page == "Full Report":

    st.title("Full Benchmark Report")
    st.caption("Raw JSON report from the latest benchmark run")
    st.divider()

    st.json(report)
    # st.json() displays a formatted, collapsible JSON viewer

    # download button
    st.download_button(
        label="Download JSON Report",
        data=json.dumps(report, indent=2, ensure_ascii=False),
        file_name=Path(latest_report).name,
        mime="application/json",
        use_container_width=True
    )
'''

Path("dashboard.py").write_text(dashboard_code.strip(), encoding="utf-8")
print("Created: dashboard.py")
print(f"File size: {Path('dashboard.py').stat().st_size:,} bytes")

Created: dashboard.py
File size: 9,826 bytes


In [23]:
import subprocess
import sys

proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "dashboard.py"],
)
print("Dashboard starting...")
print("Go to: http://localhost:8501")

Dashboard starting...
Go to: http://localhost:8501


In [24]:
from pathlib import Path

# .gitignore
gitignore = """
# ChromaDB database
data/chroma_db/

# benchmark reports
reports/

# Jupyter checkpoints
.ipynb_checkpoints/

# Python cache
__pycache__/
*.pyc

# environment
.env
*.env
"""
Path(".gitignore").write_text(gitignore.strip(), encoding="utf-8")
print("Created: .gitignore")

# requirements.txt
requirements = """streamlit
pymupdf
groq
google-genai
pydantic>=2.0
chromadb
sentence-transformers
"""
Path("requirements.txt").write_text(requirements.strip(), encoding="utf-8")
print("Created: requirements.txt")

# README.md
readme = """# LLMArena — Multi-LLM Research Paper Extraction Benchmarker

> Benchmark Gemini vs Llama 3 on extracting structured data from academic PDFs

## What it does
- Extracts 13 structured fields from any research paper PDF
- Benchmarks Gemini 2.5 Flash vs Llama 3 (Groq) on accuracy, speed and cost
- RAG pipeline using ChromaDB — ask natural language questions across papers
- Streamlit dashboard with leaderboard, charts and Q&A viewer

## Results
| Model | Accuracy | Latency | Cost |
|-------|----------|---------|------|
| Llama 3 (Groq) | 100% | 0.86s | Free |
| Gemini 2.5 Flash | 100% | 12.79s | $0.0007 |

## Tech stack
Python 3.11 · PyMuPDF · Groq · Gemini · ChromaDB · Pydantic · Streamlit

## Run locally
```bash
pip install -r requirements.txt
jupyter notebook LLMArena.ipynb
streamlit run dashboard.py
```

## Key finding
Llama 3 via Groq is 14.9x faster than Gemini and completely free — with identical accuracy.
"""
Path("README.md").write_text(readme.strip(), encoding="utf-8")
print("Created: README.md")

Created: .gitignore
Created: requirements.txt
Created: README.md


In [25]:
gitignore = """
# ChromaDB database
data/chroma_db/

# benchmark reports
reports/

# Jupyter checkpoints
.ipynb_checkpoints/

# Python cache
__pycache__/
*.pyc

# environment
.env
*.env

# notebooks — contain API keys
*.ipynb
"""
Path(".gitignore").write_text(gitignore.strip(), encoding="utf-8")
print("Updated: .gitignore")

Updated: .gitignore
